<a href="https://colab.research.google.com/github/fourmodern/2026_aidrugdiscovery/blob/main/Day06_LLM_Agent/t110_esm2_peptide_optimization_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. 실행 환경 / 요구사항

- **GPU 권장, CPU도 가능.** Colab: 런타임 > 런타임 유형 변경 > T4 GPU.
- 내려받는 모델
  - `ChatterjeeLab/PepMLM-650M` (약 2.6GB) — gated 아님, HF 로그인 불필요
  - `facebook/esm2_t6_8M_UR50D` (약 30MB) — EvoProtGrad 전문가 모델
- 예상 소요 시간: GPU 3~5분 / CPU 8~12분 (대부분 모델 다운로드 시간)

In [ ]:
# 실행 환경 확인 (Colab: 런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU)
import torch

print("PyTorch:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[안내] CPU 런타임입니다. 실행은 되지만 느립니다.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)


# ESM-2 Protein-Peptide Binding Optimization Tutorial

This notebook demonstrates how to use ESM-2, a protein language model from Facebook AI Research, to generate and optimize peptide binders for target proteins. In this lecture version the example target is **PDE5A** (phosphodiesterase type 5A; UniProt **O76074**), the target of PDE5 inhibitors such as sildenafil and tadalafil. The workflow includes:
1. Setting up the environment
2. Loading the ESM-2 model
3. Preparing a protein sequence
4. Generating peptide sequences
5. Optimizing binding affinity with evolutionary strategies

---


In [ ]:
# Step 1: Setup
# 필요한 라이브러리 설치
#
# 주의: PyPI의 "esm" 패키지는 EvolutionaryScale의 ESM3입니다(ESM-2가 아님).
#       이 노트북은 transformers로 ESM-2 계열 모델을 불러오므로 esm 패키지가 필요 없습니다.
#       torch는 Colab에 이미 설치되어 있어 재설치하지 않습니다.
!pip install -q transformers

### Step 1: Environment Setup

`transformers` 만 설치하면 됩니다. `torch` 는 Colab에 이미 있고, PyPI의 `esm` 패키지는
**ESM-2가 아니라 ESM3**(EvolutionaryScale)이므로 이 노트북에는 필요 없습니다.
ESM-2 계열 가중치는 Hugging Face Hub에서 `transformers` 로 직접 불러옵니다.
(ESM3 실습은 별도 노트북 `t111_esm3_protein_design.ipynb` 를 참고하세요.)

In [ ]:
# Step 2: Load the PepMLM (ESM-2 650M 기반) model
# (PepMLM 저장소가 TianlaiChen -> ChatterjeeLab 으로 이전되었습니다)
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import pandas as pd
import numpy as np
from torch.distributions import Categorical

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the pre-trained ESM-2(PepMLM) model and tokenizer from Hugging Face
# 가중치 약 2.6GB. 라이선스 동의/로그인 불필요(gated 아님). 첫 실행 시 다운로드에 수 분 소요.
model = AutoModelForMaskedLM.from_pretrained("ChatterjeeLab/PepMLM-650M").to(device).eval()
tokenizer = AutoTokenizer.from_pretrained("ChatterjeeLab/PepMLM-650M")

print("Model and tokenizer loaded successfully. device =", model.device)


### Step 2: Load ESM-2 Model
We use a pre-trained ESM-2 model available on Hugging Face's model hub. ESM-2 is designed for understanding protein sequences, making it suitable for predicting peptide interactions with target proteins.


In [ ]:

# Step 3: Prepare a protein sequence
# 표적 단백질: PDE5A (PDE5 저해제 sildenafil/tadalafil의 표적) - UniProt O76074, canonical 875 aa
protein_seq = "MERAGPSFGQQRQQQQPQQQKQQQRDQDSVEAWLDDHWDFTFSYFVRKATREMVNAWFAERVHTIPVCKEGIRGHTESCSCPLQQSPRADNSAPGTPTRKISASEFDRPLRPIVVKDSEGTVSFLSDSEKKEQMPLTPPRFDHDEGDQCSRLLELVKDISSHLDVTALCHKIFLHIHGLISADRYSLFLVCEDSSNDKFLISRLFDVAEGSTLEEVSNNCIRLEWNKGIVGHVAALGEPLNIKDAYEDPRFNAEVDQITGYKTQSILCMPIKNHREEVVGVAQAINKKSGNGGTFTEKDEKDFAAYLAFCGIVLHNAQLYETSLLENKRNQVLLDLASLIFEEQQSLEVILKKIAATIISFMQVQKCTIFIVDEDCSDSFSSVFHMECEELEKSSDTLTREHDANKINYMYAQYVKNTMEPLNIPDVSKDKRFPWTTENTGNVNQQCIRSLLCTPIKNGKKNKVIGVCQLVNKMEENTGKVKPFNRNDEQFLEAFVIFCGLGIQNTQMYEAVERAMAKQMVTLEVLSYHASAAEEETRELQSLAAAVVPSAQTLKITDFSFSDFELSDLETALCTIRMFTDLNLVQNFQMKHEVLCRWILSVKKNYRKNVAYHNWRHAFNTAQCMFAALKAGKIQNKLTDLEILALLIAALSHDLDHRGVNNSYIQRSEHPLAQLYCHSIMEHHHFDQCLMILNSPGNQILSGLSIEEYKTTLKIIKQAILATDLALYIKRRGEFFELIRKNQFNLEDPHQKELFLAMLMTACDLSAITKPWPIQQRIAELVATEFFDQGDRERKELNIEPTDLMNREKKNKIPSMQVGFIDAICLQLYEALTHVSEDCFPLLDGCRKNRQKWQALAEQQEKMLINGESGQAKRN"

# Tokenize the protein sequence for the model
inputs = tokenizer(protein_seq, return_tensors="pt")

print("Protein sequence tokenized.")
print("Inputs:", inputs)



### Step 3: Generate Peptide Sequence
Using PepMLM (Masked Language Modeling), we can generate a peptide sequence that is predicted to bind the input protein. The model predicts the most suitable peptide based on the given protein sequence.


In [ ]:

def compute_pseudo_perplexity(model, tokenizer, protein_seq, binder_seq):
    sequence = protein_seq + binder_seq
    original_input = tokenizer.encode(sequence, return_tensors='pt').to(model.device)
    length_of_binder = len(binder_seq)

    # Prepare a batch with each row having one masked token from the binder sequence
    masked_inputs = original_input.repeat(length_of_binder, 1)
    positions_to_mask = torch.arange(-length_of_binder - 1, -1, device=model.device)

    masked_inputs[torch.arange(length_of_binder), positions_to_mask] = tokenizer.mask_token_id

    # Prepare labels for the masked tokens
    labels = torch.full_like(masked_inputs, -100)
    labels[torch.arange(length_of_binder), positions_to_mask] = original_input[0, positions_to_mask]

    # Get model predictions and calculate loss
    with torch.no_grad():
        outputs = model(masked_inputs, labels=labels)
        loss = outputs.loss

    # Loss is already averaged by the model
    avg_loss = loss.item()
    pseudo_perplexity = np.exp(avg_loss)
    return pseudo_perplexity


def generate_peptide_for_single_sequence(protein_seq, peptide_length = 15, top_k = 3, num_binders = 4):

    peptide_length = int(peptide_length)
    top_k = int(top_k)
    num_binders = int(num_binders)

    binders_with_ppl = []

    for _ in range(num_binders):
        # Generate binder
        masked_peptide = '<mask>' * peptide_length
        input_sequence = protein_seq + masked_peptide
        inputs = tokenizer(input_sequence, return_tensors="pt").to(model.device)

        with torch.no_grad():
            logits = model(**inputs).logits
        mask_token_indices = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
        logits_at_masks = logits[0, mask_token_indices]

        # Apply top-k sampling
        top_k_logits, top_k_indices = logits_at_masks.topk(top_k, dim=-1)
        probabilities = torch.nn.functional.softmax(top_k_logits, dim=-1)
        predicted_indices = Categorical(probabilities).sample()
        predicted_token_ids = top_k_indices.gather(-1, predicted_indices.unsqueeze(-1)).squeeze(-1)

        generated_binder = tokenizer.decode(predicted_token_ids, skip_special_tokens=True).replace(' ', '')

        # Compute PPL for the generated binder
        ppl_value = compute_pseudo_perplexity(model, tokenizer, protein_seq, generated_binder)

        # Add the generated binder and its PPL to the results list
        binders_with_ppl.append([generated_binder, ppl_value])

    return binders_with_ppl

def generate_peptide(input_seqs, peptide_length=15, top_k=3, num_binders=4):
    if isinstance(input_seqs, str):  # Single sequence
        binders = generate_peptide_for_single_sequence(input_seqs, peptide_length, top_k, num_binders)
        return pd.DataFrame(binders, columns=['Binder', 'Pseudo Perplexity'])

    elif isinstance(input_seqs, list):  # List of sequences
        results = []
        for seq in input_seqs:
            binders = generate_peptide_for_single_sequence(seq, peptide_length, top_k, num_binders)
            for binder, ppl in binders:
                results.append([seq, binder, ppl])
        return pd.DataFrame(results, columns=['Input Sequence', 'Binder', 'Pseudo Perplexity'])

### Step 4: Generate Binders and Rank by Pseudo-Perplexity

위에서 정의한 함수로 15-mer 펩타이드 후보를 top-k 샘플링으로 여러 개 만들고,
**pseudo-perplexity (PPL)** 로 순위를 매깁니다.
PPL은 binder의 각 잔기를 하나씩 마스킹해 모델이 원래 잔기를 얼마나 잘 복원하는지를 재는 값으로,
**낮을수록** 모델이 그 서열을 "그럴듯하다"고 본다는 뜻입니다.
(주의: PPL은 실제 결합 친화도(Kd/IC50)의 예측치가 아닙니다. 실험 검증이 반드시 필요합니다.)

In [ ]:
# 실행 시간 안내:
#  - GPU(T4): 전체 5개 binder 생성에 약 20~30초
#  - CPU    : binder 1개당 약 40초 (5개면 3~4분). 느리면 num_binders를 2로 줄이세요.
results_df = generate_peptide(protein_seq, peptide_length=15, top_k=3, num_binders=5)
print(results_df.sort_values("Pseudo Perplexity").to_string(index=False))
print()
print("Pseudo-perplexity(PPL)가 낮을수록 ESM-2가 '그럴듯하다'고 보는 서열입니다.")
print("주의: PPL은 결합력(Kd)의 직접 예측치가 아니라 언어모델 관점의 서열 그럴듯함 지표입니다.")

## Step 5: In Silico Directed Evolution of the Peptide Binder with EvoProtGrad and ESM-2

EvoProtGrad는 gradient 기반 discrete MCMC로 서열을 탐색하는 in silico 방향진화(directed evolution)
라이브러리입니다. 여기서는 작은 ESM-2(`facebook/esm2_t6_8M_UR50D`)를 "전문가(expert)" 모델로 쓰고,
표적 단백질과 링커 구간은 `preserved_regions` 로 고정한 채 **뒤쪽 15-mer 펩타이드만** 변이시킵니다.

> `scoring_strategy='mutant_marginal'` 을 씁니다. EvoProtGrad의 기본값인 `pseudolikelihood_ratio`
> 는 최신 transformers에서 텐서 차원 불일치로 실패합니다.

In [ ]:
# evo_prot_grad 설치
#
# 두 가지 주의사항이 있습니다.
# 1) !pip install evo_prot_grad>=0.1.0 처럼 쓰면 셸이 ">"를 리다이렉션으로 해석해
#    버전 조건이 무시되고 "=0.1.0" 이라는 파일이 생깁니다. 반드시 따옴표로 묶습니다.
# 2) evo_prot_grad 0.2는 transformers==4.38.0 을 정확히 고정하고 있어서 그대로 설치하면
#    transformers가 2024년 버전으로 강제 다운그레이드되고 앞 단계가 깨집니다.
#    --no-deps 로 설치해 현재 transformers를 유지합니다.
#    (--no-deps 로 설치해도 evo_prot_grad 0.2 는 최신 transformers에서 정상 동작합니다.)
!pip install -q --no-deps "evo_prot_grad>=0.2"

import torch
import evo_prot_grad
from transformers import AutoTokenizer, EsmForMaskedLM

print("evo_prot_grad import OK")

In [ ]:
def run_evo_prot_grad_on_paired_sequence(paired_protein_sequence):
    # Replace ':' with a string of 20 'G' amino acids
    separator = 'G' * 20
    sequence_with_separator = paired_protein_sequence.replace(':', separator)

    # Determine the start and end indices of the first protein and the separator
    separator_start_index = sequence_with_separator.find(separator)
    first_protein_end_index = separator_start_index
    separator_end_index = separator_start_index + len(separator)

    # Format the sequence into FASTA format
    fasta_format_sequence = f">Paired_Protein_Sequence\n{sequence_with_separator}"

    # Save the sequence to a temporary file
    temp_fasta_path = "temp_paired_sequence.fasta"
    with open(temp_fasta_path, "w") as file:
        file.write(fasta_format_sequence)

    # Use a smaller ESM-2 model to reduce memory usage
    esm2_expert = evo_prot_grad.get_expert(
        'esm',
        model=EsmForMaskedLM.from_pretrained("facebook/esm2_t6_8M_UR50D"),  # Use a smaller ESM-2 model
        tokenizer=AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D"),
        # pseudolikelihood_ratio는 최신 transformers에서 텐서 차원 불일치로 실패합니다.
        # mutant_marginal은 EvoProtGrad 논문에서 권장하는 전략이며 정상 동작합니다.
        scoring_strategy='mutant_marginal',
        temperature=0.95,
        device=device  # 위에서 정한 'cuda' 또는 'cpu'
    )

    # Initialize wildtype sequence for the expert with the correct format
    wildtype_sequence = sequence_with_separator.replace(" ", "")  # Make sure the input sequence has no spaces
    esm2_expert.init_wildtype(wildtype_sequence)

    # Initialize Directed Evolution with the preserved first protein and separator region
    directed_evolution = evo_prot_grad.DirectedEvolution(
        wt_fasta=temp_fasta_path,
        output='best',
        experts=[esm2_expert],
        parallel_chains=1,  # Reduce parallel chains to save memory
        n_steps=50,
        max_mutations=15,
        verbose=True,
        preserved_regions=[(0, first_protein_end_index), (separator_start_index, separator_end_index)]
    )

    # Run the evolution process
    variants, scores = directed_evolution()

    # Process the results and split them into Protein 1 and Protein 2
    for variant, score in zip(variants, scores):
        # Remove spaces from the sequence
        evolved_sequence_no_spaces = variant.replace(" ", "")

        # Split the sequence at the separator
        protein_1, protein_2 = evolved_sequence_no_spaces.split(separator)

        print(f"Protein: {protein_1}, Evolved Peptide: {protein_2}, Score: {score}")

In [ ]:
# 전문가 모델로는 작은 facebook/esm2_t6_8M_UR50D(8M)를 쓰므로 메모리 부담이 적습니다.
# n_steps=50, max_mutations=15, parallel_chains=1 기준으로 CPU에서도 1분 내외에 끝납니다.
# preserved_regions 로 표적(PDE5A)과 GGG...G 링커는 고정하고, 뒤쪽 15-mer 펩타이드만 진화시킵니다.
# Example usage
paired_protein_sequence = "MERAGPSFGQQRQQQQPQQQKQQQRDQDSVEAWLDDHWDFTFSYFVRKATREMVNAWFAERVHTIPVCKEGIRGHTESCSCPLQQSPRADNSAPGTPTRKISASEFDRPLRPIVVKDSEGTVSFLSDSEKKEQMPLTPPRFDHDEGDQCSRLLELVKDISSHLDVTALCHKIFLHIHGLISADRYSLFLVCEDSSNDKFLISRLFDVAEGSTLEEVSNNCIRLEWNKGIVGHVAALGEPLNIKDAYEDPRFNAEVDQITGYKTQSILCMPIKNHREEVVGVAQAINKKSGNGGTFTEKDEKDFAAYLAFCGIVLHNAQLYETSLLENKRNQVLLDLASLIFEEQQSLEVILKKIAATIISFMQVQKCTIFIVDEDCSDSFSSVFHMECEELEKSSDTLTREHDANKINYMYAQYVKNTMEPLNIPDVSKDKRFPWTTENTGNVNQQCIRSLLCTPIKNGKKNKVIGVCQLVNKMEENTGKVKPFNRNDEQFLEAFVIFCGLGIQNTQMYEAVERAMAKQMVTLEVLSYHASAAEEETRELQSLAAAVVPSAQTLKITDFSFSDFELSDLETALCTIRMFTDLNLVQNFQMKHEVLCRWILSVKKNYRKNVAYHNWRHAFNTAQCMFAALKAGKIQNKLTDLEILALLIAALSHDLDHRGVNNSYIQRSEHPLAQLYCHSIMEHHHFDQCLMILNSPGNQILSGLSIEEYKTTLKIIKQAILATDLALYIKRRGEFFELIRKNQFNLEDPHQKELFLAMLMTACDLSAITKPWPIQQRIAELVATEFFDQGDRERKELNIEPTDLMNREKKNKIPSMQVGFIDAICLQLYEALTHVSEDCFPLLDGCRKNRQKWQALAEQQEKMLINGESGQAKRN:FDEDDPLAPRLLEEE"  # 표적 PDE5A (UniProt O76074) : 초기 15-mer 펩타이드 바인더
run_evo_prot_grad_on_paired_sequence(paired_protein_sequence)